# Lab 17-01 · Measuring Random Processes

Five parts, each built on `randproc.py`. **Predict first**, then run. The handout is [README.md](README.md); Part 6 (ngspice) runs outside the notebook.

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import randproc as rp
rng = np.random.default_rng(1701)

## Part 1 · Ensemble against time averages

*Predict:* $R[0]$ and $R[1]$ of the AR(1) process $x[n] = 0.9x[n-1] + w[n]$ with $\sigma_w^2 = 1$, and whether one long record and 2000 short records agree.

In [2]:
a, var_w = 0.9, 1.0
sigma2 = var_w / (1 - a * a)          # R[0] = 5.263
tau_c = -1 / math.log(a)              # in samples: the AR(1) is a sampled exponential with a = exp(-1/tau_c)
X = np.array([rp.ou_samples(300, sigma2, tau_c, 1.0, rng) for _ in range(2000)])
R_ens = rp.autocorr_ensemble(X, 150, 5)
R_time = rp.autocorr_time(rp.ou_samples(1 << 18, sigma2, tau_c, 1.0, rng), 5)
R_th = sigma2 * a ** np.arange(6)
for k in range(6):
    print(f"k = {k}: theory {R_th[k]:.3f}  ensemble {R_ens[k]:.3f}  time average {R_time[k]:.3f}")
assert np.all(np.abs(R_ens / R_th - 1) < 0.08) and np.all(np.abs(R_time / R_th - 1) < 0.04)
assert round(R_th[1], 2) == 4.74

k = 0: theory 5.263  ensemble 5.154  time average 5.276
k = 1: theory 4.737  ensemble 4.718  time average 4.748
k = 2: theory 4.263  ensemble 4.209  time average 4.271
k = 3: theory 3.837  ensemble 3.668  time average 3.840
k = 4: theory 3.453  ensemble 3.312  time average 3.452
k = 5: theory 3.108  ensemble 3.005  time average 3.103


## Part 2 · Noise through a one-pole filter (Example 17-01.2)

*Predict:* the rms output for 10 nV/√Hz white noise through a 1 MHz one-pole filter, from the noise bandwidth $\pi f_{3\text{dB}}/2$; and the correlation coefficient at 100 ns.

In [3]:
N0 = (10e-9) ** 2                      # one-sided density, V^2/Hz
RC = 1 / (2 * math.pi * 1e6)
BN = 1 / (4 * RC)
print(f"noise bandwidth {BN / 1e6:.4f} MHz, predicted rms {math.sqrt(N0 * BN) * 1e6:.2f} uV")
# (i) white noise sampled at 1 GHz through the impulse-invariant one-pole filter
fs = 1e9
x = rp.white_noise(1 << 20, N0 / 2, fs, rng)
ap = math.exp(-1 / (fs * RC))
from scipy import signal
y = signal.lfilter([1 - ap], [1, -ap], x)[5000:]
# (ii) the exact sampled process (AR(1), section 3.7)
z = rp.ou_samples(1 << 18, N0 / (4 * RC), RC, 1e-8, rng)
rho_y = rp.autocorr_time(y, 100)[100] / np.var(y)
rho_z = rp.autocorr_time(z, 10)[10] / np.var(z)
print(f"filtered white noise: rms {np.std(y) * 1e6:.2f} uV, correlation at 100 ns {rho_y:.3f}")
print(f"exact AR(1) samples : rms {np.std(z) * 1e6:.2f} uV, correlation at 100 ns {rho_z:.3f}  (theory {math.exp(-1e-7 / RC):.3f})")
assert abs(np.std(y) / 12.53e-6 - 1) < 0.03 and abs(np.std(z) / 12.53e-6 - 1) < 0.02
assert abs(rho_y - 0.533) < 0.03 and abs(rho_z - 0.533) < 0.03

noise bandwidth 1.5708 MHz, predicted rms 12.53 uV
filtered white noise: rms 12.66 uV, correlation at 100 ns 0.541
exact AR(1) samples : rms 12.55 uV, correlation at 100 ns 0.533  (theory 0.533)


## Part 3 · Spectra of digital signals

*Predict:* where the nulls of polar NRZ fall, how much of the unipolar signal's power sits in its DC line, and where the half-sine pulse has its first null.

In [4]:
sps, nbits = 16, 1 << 16
fsym = float(sps)                      # T = 1, so frequencies are in units of the bit rate
bits = rng.integers(0, 2, nbits)
polar = rp.pam_waveform(2.0 * bits - 1.0, sps)
unipolar = rp.pam_waveform(1.0 * bits, sps)
halfsine = rp.pam_waveform(2.0 * bits - 1.0, sps, pulse="halfsine")
f, Gp = rp.psd_welch(polar, fsym, nperseg=2048)
_, Gh = rp.psd_welch(halfsine, fsym, nperseg=2048)
df = f[1] - f[0]
main = np.sum(Gp[f < 1.0]) * df / (np.sum(Gp) * df)
dc_share = np.mean(unipolar) ** 2 / np.mean(unipolar ** 2)
sel = (f > 1.2) & (f < 1.8)
null_hs = f[sel][np.argmin(Gh[sel])]
print(f"polar NRZ: fraction of power below the bit rate {main:.3f} (theory {rp.nrz_power_fraction(1):.3f})")
print(f"unipolar NRZ: DC-line share of power {dc_share:.3f} (theory {rp.pam_line_power(0.5, 0.25):.3f})")
print(f"half-sine first null at fT = {null_hs:.3f}")
plt.semilogy(f, Gp / 2, label="polar NRZ")
plt.semilogy(f, rp.nrz_psd(f, 1.0, 1.0), "--", lw=0.8, label="A^2 T sinc^2(fT)")
plt.semilogy(f, Gh / 2, label="half-sine")
plt.xlim(0, 4); plt.ylim(1e-5, 2); plt.xlabel("fT"); plt.ylabel("two-sided PSD"); plt.legend(); plt.show()
assert abs(main - 0.903) < 0.01 and abs(dc_share - 0.5) < 0.01 and abs(null_hs - 1.5) < 0.02

polar NRZ: fraction of power below the bit rate 0.904 (theory 0.903)
unipolar NRZ: DC-line share of power 0.500 (theory 0.500)
half-sine first null at fT = 1.500


## Part 4 · Noise folding (Example 17-01.5)

*Predict:* how much the in-band density rises when noise with $\tau_c$ = 0.1 µs, and then 1 µs, is sampled at 1 MS/s without an anti-alias filter.

In [5]:
sig2, fs = 1e-6, 1e6
T = 1 / fs
for tc in (0.1e-6, 1e-6):
    a = math.exp(-T / tc)
    closed = (1 + a) / (1 - a) * T / (2 * tc)
    alias = rp.aliased_density([0.0], sig2, tc, fs, terms=20000)[0] / rp.lorentzian_two_sided(0.0, sig2, tc)
    x = rp.ou_samples(1 << 18, sig2, tc, T, rng)
    f, G = rp.psd_welch(x, fs, nperseg=1024)
    band = (f > 0) & (f < 50e3)
    G_fold = 2 * rp.aliased_density(f[band], sig2, tc, fs, terms=4000)
    welch = np.mean(G[band]) / (2 * rp.lorentzian_two_sided(0.0, sig2, tc))  # sampled density / continuous G(0)
    # the alias sum is truncated at 20000 terms each side; its 1/m^2 tail leaves an error in the fifth digit
    print(f"tau_c = {tc * 1e6:.1f} us: closed form {closed:.4f}, alias sum {alias:.4f}, Welch near DC {welch:.3f}; sample power {np.var(x):.3e} V^2")
    assert abs(closed - alias) < 1e-4 * closed and abs(np.var(x) / sig2 - 1) < 0.05
    assert abs(np.mean(G[band]) / np.mean(G_fold) - 1) < 0.05
    assert abs(round(closed, 2 if tc < 0.5e-6 else 3) - (5.00 if tc < 0.5e-6 else 1.082)) < 1e-9

tau_c = 0.1 us: closed form 5.0005, alias sum 5.0002, Welch near DC 5.053; sample power 1.003e-06 V^2
tau_c = 1.0 us: closed form 1.0820, alias sum 1.0820, Welch near DC 1.058; sample power 9.977e-07 V^2


## Part 5 · In-phase and quadrature noise

*Predict:* the powers of $n_I$ and $n_Q$ relative to the bandpass noise, and their correlation at lag 0 and at other lags, (a) with a 20 MHz band centred on the 50 MHz oscillator, (b) with a 10 MHz band from 50 to 60 MHz (all on one side).

In [6]:
fs = 200e6
# (a) symmetric band
x = rp.bandpass_noise(1 << 19, fs, 50e6, 20e6, 1.0, rng)
nI, nQ = rp.iq_components(x, fs, 50e6, 20e6)
rho = lambda k: np.mean(nI[k:] * nQ[: len(nQ) - k]) / math.sqrt(np.var(nI) * np.var(nQ))
print(f"(a) P_I / P = {np.var(nI) / np.var(x):.3f}, P_Q / P = {np.var(nQ) / np.var(x):.3f}, rho(0) = {rho(0):+.4f}, max |rho(k)| for k <= 20: {max(abs(rho(k)) for k in range(21)):.3f}")
assert abs(np.var(nI) / np.var(x) - 1) < 0.03 and abs(np.var(nQ) / np.var(x) - 1) < 0.03
assert abs(rho(0)) < 0.02 and max(abs(rho(k)) for k in range(21)) < 0.03
# (b) one-sided band, oscillator at its lower edge: theory rho(tau) = -(1 - cos x)/x, x = 2 pi B tau
B = 10e6
x = rp.bandpass_noise(1 << 19, fs, 55e6, B, 1.0, rng)
nI, nQ = rp.iq_components(x, fs, 50e6, 2 * B)
ks = np.arange(0, 21)
meas = np.array([rho(int(k)) for k in ks])
xx = 2 * np.pi * B * ks / fs
theory = np.where(ks == 0, 0.0, -(1 - np.cos(xx)) / np.where(xx == 0, 1, xx))
for k in (0, 3, 7, 10, 20):
    print(f"(b) lag {k:2d} ({k / fs * 1e9:5.1f} ns): measured {meas[k]:+.3f}, theory {theory[k]:+.3f}")
print(f"(b) P_I / P = {np.var(nI) / np.var(x):.3f}; most negative correlation {meas.min():.3f} at lag {ks[np.argmin(meas)]} (theory -0.725 at 0.371/B = {0.371 / B * 1e9:.1f} ns)")
plt.plot(ks / fs * 1e9, meas, "o", label="measured"); plt.plot(ks / fs * 1e9, theory, label="-(1 - cos x)/x")
plt.xlabel("lag (ns)"); plt.ylabel("correlation of n_I(t+tau), n_Q(t)"); plt.legend(); plt.show()
assert abs(meas[0]) < 0.02 and np.max(np.abs(meas - theory)) < 0.05 and abs(np.var(nI) / np.var(x) - 1) < 0.03

(a) P_I / P = 1.002, P_Q / P = 1.000, rho(0) = -0.0063, max |rho(k)| for k <= 20: 0.009


(b) lag  0 (  0.0 ns): measured +0.000, theory +0.000
(b) lag  3 ( 15.0 ns): measured -0.439, theory -0.437
(b) lag  7 ( 35.0 ns): measured -0.729, theory -0.722
(b) lag 10 ( 50.0 ns): measured -0.649, theory -0.637
(b) lag 20 (100.0 ns): measured +0.003, theory -0.000
(b) P_I / P = 1.000; most negative correlation -0.729 at lag 7 (theory -0.725 at 0.371/B = 37.1 ns)


## Part 6 · ngspice

Run `python tools/run_spice.py 17-01` from the repository root. Both resistors give $\sqrt{kT/C}$ = 2.035 µV across the 1 nF capacitor: the resistor sets the shape of the noise spectrum, but the total is fixed by equipartition.